# Media Framing — Multi-Model Test Run

**Model-selection sandbox** for Mechanism 2. Runs the updated media-framing prompt on the 100-row manually annotated test sample across several LLMs to inform the final production model choice (`gpt-5.4-mini`). The new and old prompt snapshots are kept side by side. Outputs:

- `framing_annotation_sample_100_multimodel_labels_long.csv` — one row per article × entity × model × prompt.
- `framing_annotation_sample_100_multimodel_labels_wide.csv` — one row per sample item with one label/evidence/status column per model.
- `framing_annotation_sample_100_multimodel_run_summary.csv` — completion and hit-rate diagnostics per model.

API keys are read only from environment variables or a local `.env` file. No secrets are stored in the notebook.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import re
import time
import urllib.error
import urllib.request

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 220)

# Inputs and outputs (paths simplified for transparency, not for re-execution).
INPUT_CSV_PATH = Path("framing_annotation_sample_100_test.csv")
OLD_PROMPT_PATH = Path("framing_codebook_prompt_old_reference.txt")
NEW_PROMPT_PATH = Path("framing_codebook_prompt_new_test.txt")

LONG_OUTPUT_PATH = Path("framing_annotation_sample_100_multimodel_labels_long.csv")
WIDE_OUTPUT_PATH = Path("framing_annotation_sample_100_multimodel_labels_wide.csv")
SUMMARY_OUTPUT_PATH = Path("framing_annotation_sample_100_multimodel_run_summary.csv")


In [ ]:
# Configuration
# Keep the old prompt available for A/B testing. Default is the new tuned prompt only.
PROMPT_VARIANTS_TO_RUN = ['new']  # use ['old', 'new'] if you want prompt A/B testing

# Run is disabled by default to prevent accidental API spend.
RUN_API = True  # Final test run: calls the APIs. Set False for dry-run validation only.
SLEEP_SECONDS_BETWEEN_CALLS = 12  # Claude has a low input-token/minute limit; keep this slow.
SAVE_EVERY_N_CALLS = 1
MAX_RATE_LIMIT_RETRIES = 5
RATE_LIMIT_RETRY_SECONDS = 65
STOP_ON_ERROR = False  # Continue and log errors; reruns can resume missing/non-ok rows.
RESET_EXISTING_RESULTS = False  # Resume existing ok rows instead of overwriting completed labels.
# Do not revert this notebook state while resuming: the existing long CSV already contains completed labels.
MAX_ITEMS_PER_MODEL = None  # Set to 1 for a low-cost smoke test before the full 100 rows.

# Model IDs are intentionally centralized here. Edit if your API account exposes different IDs.
MODEL_RUNS = [
    # User-requested comparison set. Keep RUN_API=False until you are ready to execute.
    # gpt-5-mini is the GPT-5 mini model id; gpt-5o-mini returned model_not_found in the API.
    {'provider': 'openai', 'model': 'gpt-5.4', 'enabled': True},
    {'provider': 'openai', 'model': 'gpt-5.4-mini', 'enabled': True},
    {'provider': 'openai', 'model': 'gpt-5-mini', 'enabled': True},
    # Anthropic billing smoke-test passed; keep enabled for the full multi-model comparison.
    {'provider': 'anthropic', 'model': 'claude-sonnet-4-5', 'enabled': True},
]

CATEGORIES = [
    'POSITIONS-/PARTEILICHKEITS-BIAS',
    'VERZERRUNG/MANIPULATION',
    'DISINFORMATION/FALSCHDARSTELLUNG',
    'VERSAGEN/INKOMPETENZ',
    'NEUTRAL',
    'IRRELEVANT',
]

print('Enabled model runs:')
for cfg in MODEL_RUNS:
    if cfg['enabled']:
        print('-', cfg['provider'], cfg['model'])

In [ ]:
sample_df = pd.read_csv(INPUT_CSV_PATH)
old_prompt = OLD_PROMPT_PATH.read_text(encoding='utf-8')
new_prompt = NEW_PROMPT_PATH.read_text(encoding='utf-8')

required_cols = {'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window'}
missing_cols = sorted(required_cols - set(sample_df.columns))
if missing_cols:
    raise KeyError(f'Missing expected columns in test CSV: {missing_cols}')

for prompt_name, prompt_text in {'old': old_prompt, 'new': new_prompt}.items():
    for placeholder in ['{context}', '{entity_mention}']:
        if placeholder not in prompt_text:
            raise ValueError(f'{prompt_name} prompt is missing placeholder {placeholder}')

print('Rows:', len(sample_df))
print('Columns:', list(sample_df.columns))
print('Old prompt chars:', len(old_prompt))
print('New prompt chars:', len(new_prompt))
display(sample_df.head(5))

In [ ]:
PROMPTS = {'old': old_prompt, 'new': new_prompt}

RESPONSE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'category': {'type': 'string', 'enum': CATEGORIES},
        'evidence': {'type': 'string'},
    },
    'required': ['category', 'evidence'],
}

SYSTEM_MESSAGE = (
    'You are a careful media-framing annotation assistant. '
    'Return exactly one JSON object with keys category and evidence. '
    'Do not add explanations or markdown.'
)


def render_prompt(template: str, row: pd.Series) -> str:
    return template.format(
        context=str(row['context_window']),
        entity_mention=str(row['entity_mention']),
    )


def safe_col_token(value: str) -> str:
    value = str(value).lower()
    value = re.sub(r'[^a-z0-9]+', '_', value).strip('_')
    return value or 'unknown'


def model_label(provider: str, model: str) -> str:
    return f'{provider}::{model}'


def run_label(provider: str, model: str, prompt_variant: str) -> str:
    return f'{prompt_variant}__{provider}__{safe_col_token(model)}'


request_rows = []
run_sample_df = sample_df if MAX_ITEMS_PER_MODEL is None else sample_df.head(int(MAX_ITEMS_PER_MODEL))
print('Rows per model for this run:', len(run_sample_df))
for prompt_variant in PROMPT_VARIANTS_TO_RUN:
    prompt_template = PROMPTS[prompt_variant]
    for cfg in MODEL_RUNS:
        if not cfg.get('enabled', False):
            continue
        for _, row in run_sample_df.iterrows():
            request_rows.append({
                'item_id': row['item_id'],
                'hit_id': row['hit_id'],
                'row_id': row['row_id'],
                'outlet': row['outlet'],
                'article_title': row['article_title'],
                'entity_mention': row['entity_mention'],
                'context_window': row['context_window'],
                'prompt_variant': prompt_variant,
                'provider': cfg['provider'],
                'model': cfg['model'],
                'model_label': model_label(cfg['provider'], cfg['model']),
                'run_label': run_label(cfg['provider'], cfg['model'], prompt_variant),
                'prompt': render_prompt(prompt_template, row),
            })

requests_df = pd.DataFrame(request_rows)
requests_df.drop(columns=['prompt']).to_csv(REQUEST_PLAN_PATH, index=False)

print('Planned calls:', len(requests_df))
print('Request plan saved to:', REQUEST_PLAN_PATH)
display(requests_df.drop(columns=['prompt']).head(10))
display(requests_df.drop(columns=['prompt']).head(10))

# No-cost technical validation: this checks local schema/plan only. It does not call any API.
expected_output_columns = [
    'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window',
    'prompt_variant', 'provider', 'model', 'model_label', 'category', 'evidence',
    'run_status', 'error', 'response_id', 'api_key_source', 'created_at_utc', 'raw_response',
]
assert not requests_df.empty, 'No requests planned. Check MODEL_RUNS and MAX_ITEMS_PER_MODEL.'
assert requests_df['prompt'].notna().all(), 'At least one rendered prompt is missing.'
assert requests_df[['item_id', 'provider', 'model', 'prompt_variant']].notna().all().all(), 'Request plan has missing ids/model metadata.'
assert requests_df.duplicated(['item_id', 'prompt_variant', 'provider', 'model']).sum() == 0, 'Duplicate model requests found.'

planned_counts = (
    requests_df.groupby(['prompt_variant', 'provider', 'model'], as_index=False)
    .size()
    .rename(columns={'size': 'planned_rows'})
)
display(planned_counts)
print('Analysis-ready long output columns will be:')
print(expected_output_columns)
print('No-cost technical validation passed: CSV, prompts, model plan, and output paths are valid.')
print('No API calls have been made by this cell.')


In [ ]:
def read_env_value(name, project_root=PROJECT_ROOT):
    # Prefer the project .env over the notebook/shell environment.
    # This avoids accidentally using a stale ANTHROPIC_API_KEY from an old Jupyter kernel.
    env_path = project_root / '.env'
    if env_path.exists():
        for line in env_path.read_text(encoding='utf-8').splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, raw_value = line.split('=', 1)
            if key.strip() == name:
                value = raw_value.strip().strip('"').strip("'")
                if value:
                    return value, str(env_path)

    value = os.environ.get(name)
    if value:
        return value, 'environment variable'
    return None, None


def post_json(url: str, headers: dict, payload: dict, timeout: int = 120) -> dict:
    data = json.dumps(payload).encode('utf-8')
    for attempt in range(MAX_RATE_LIMIT_RETRIES + 1):
        request = urllib.request.Request(url=url, data=data, headers=headers, method='POST')
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.loads(response.read().decode('utf-8'))
        except urllib.error.HTTPError as exc:
            body = exc.read().decode('utf-8', errors='replace')
            if exc.code == 429 and attempt < MAX_RATE_LIMIT_RETRIES:
                wait_seconds = RATE_LIMIT_RETRY_SECONDS * (attempt + 1)
                print(f'HTTP 429 rate limit. Waiting {wait_seconds}s before retry {attempt + 1}/{MAX_RATE_LIMIT_RETRIES}.')
                time.sleep(wait_seconds)
                continue
            raise RuntimeError(f'HTTP {exc.code}: {body}') from exc
    raise RuntimeError('Unexpected post_json retry loop exit')


def extract_json_text_from_openai(response: dict) -> str:
    if response.get('output_text'):
        return response['output_text']
    parts = []
    for item in response.get('output', []):
        for content in item.get('content', []):
            text = content.get('text') or content.get('value')
            if text:
                parts.append(text)
    return '\n'.join(parts).strip()


def extract_json_text_from_anthropic(response: dict) -> str:
    parts = []
    for block in response.get('content', []):
        if block.get('type') == 'text' and block.get('text'):
            parts.append(block['text'])
    return '\n'.join(parts).strip()


def parse_json_object(text: str) -> dict:
    text = str(text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def normalize_annotation(parsed: dict) -> dict:
    category = str(parsed.get('category', '')).strip()
    evidence = str(parsed.get('evidence', '')).strip()
    if category not in CATEGORIES:
        raise ValueError(f'Invalid category: {category!r}')
    if category in {'NEUTRAL', 'IRRELEVANT'}:
        evidence = ''
    return {'category': category, 'evidence': evidence}


def call_openai_model(model: str, prompt: str) -> dict:
    api_key, source = read_env_value('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('OPENAI_API_KEY not found. Export it in your shell or put it in a local .env file.')

    payload = {
        'model': model,
        'input': [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': prompt},
        ],
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'framing_annotation',
                'schema': RESPONSE_SCHEMA,
                'strict': True,
            }
        },
        'reasoning': {'effort': 'low'},
        'max_output_tokens': 1200,
    }
    response = post_json(
        'https://api.openai.com/v1/responses',
        headers={
            'Authorization': f'Bearer {api_key}',
            'Content-Type': 'application/json',
        },
        payload=payload,
    )
    output_text = extract_json_text_from_openai(response)
    annotation = normalize_annotation(parse_json_object(output_text))
    return {
        **annotation,
        'raw_response': output_text,
        'response_id': response.get('id', ''),
        'api_key_source': source,
    }


def call_anthropic_model(model: str, prompt: str) -> dict:
    api_key, source = read_env_value('ANTHROPIC_API_KEY')
    if not api_key:
        raise RuntimeError('ANTHROPIC_API_KEY not found. Export it in your shell or put it in a local .env file.')

    payload = {
        'model': model,
        'max_tokens': 300,
        'temperature': 0,
        'system': SYSTEM_MESSAGE,
        'messages': [{'role': 'user', 'content': prompt}],
        'tools': [
            {
                'name': 'emit_framing_annotation',
                'description': 'Return exactly one final media-framing annotation.',
                'input_schema': RESPONSE_SCHEMA,
            }
        ],
        'tool_choice': {'type': 'tool', 'name': 'emit_framing_annotation'},
    }
    response = post_json(
        'https://api.anthropic.com/v1/messages',
        headers={
            'x-api-key': api_key,
            'anthropic-version': '2023-06-01',
            'Content-Type': 'application/json',
        },
        payload=payload,
    )
    tool_inputs = [
        block.get('input')
        for block in response.get('content', [])
        if block.get('type') == 'tool_use' and block.get('name') == 'emit_framing_annotation'
    ]
    if tool_inputs:
        raw_response = json.dumps(tool_inputs[0], ensure_ascii=False)
        annotation = normalize_annotation(tool_inputs[0])
    else:
        raw_response = extract_json_text_from_anthropic(response)
        annotation = normalize_annotation(parse_json_object(raw_response))
    return {
        **annotation,
        'raw_response': raw_response,
        'response_id': response.get('id', ''),
        'api_key_source': source,
    }


def call_model(provider: str, model: str, prompt: str) -> dict:
    if provider == 'openai':
        return call_openai_model(model, prompt)
    if provider == 'anthropic':
        return call_anthropic_model(model, prompt)
    raise ValueError(f'Unsupported provider: {provider}')

for key_name in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY']:
    key_value, key_source = read_env_value(key_name)
    print(f'{key_name} available:', bool(key_value), '| source:', key_source or 'missing')

In [ ]:
def load_existing_results(path: Path) -> pd.DataFrame:
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def save_results(results, path):
    results_df = pd.DataFrame(results)
    ordered_cols = [
        'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window',
        'prompt_variant', 'provider', 'model', 'model_label', 'category', 'evidence', 'run_label',
        'run_status', 'error', 'response_id', 'api_key_source', 'created_at_utc', 'raw_response',
    ]
    existing = [col for col in ordered_cols if col in results_df.columns]
    remaining = [col for col in results_df.columns if col not in existing]
    results_df = results_df[existing + remaining]
    results_df.to_csv(path, index=False)
    return results_df


existing_df = load_existing_results(LONG_OUTPUT_PATH)
if RESET_EXISTING_RESULTS:
    existing_df = pd.DataFrame()
    print('RESET_EXISTING_RESULTS=True: ignoring any existing long-output rows.')
planned_keys = set(
    zip(
        requests_df['item_id'].astype(str),
        requests_df['prompt_variant'].astype(str),
        requests_df['provider'].astype(str),
        requests_df['model'].astype(str),
    )
)
if not existing_df.empty:
    existing_keys = list(zip(
        existing_df['item_id'].astype(str),
        existing_df['prompt_variant'].astype(str),
        existing_df['provider'].astype(str),
        existing_df['model'].astype(str),
    ))
    existing_df = existing_df.loc[[key in planned_keys for key in existing_keys]].copy()
if not existing_df.empty and 'run_status' in existing_df.columns:
    previous_non_ok = existing_df.loc[~existing_df['run_status'].eq('ok')].copy()
    if not previous_non_ok.empty:
        print(f'Dropping {len(previous_non_ok)} previous non-ok row(s) so they can be retried.')
    existing_df = existing_df.loc[existing_df['run_status'].eq('ok')].copy()
completed_keys = set()
if not existing_df.empty:
    ok_existing = existing_df.loc[existing_df.get('run_status', '') == 'ok']
    completed_keys = set(
        zip(
            ok_existing['item_id'].astype(str),
            ok_existing['prompt_variant'].astype(str),
            ok_existing['provider'].astype(str),
            ok_existing['model'].astype(str),
        )
    )

results = existing_df.to_dict('records') if not existing_df.empty else []
print('Existing result rows:', len(results))
print('Completed ok calls:', len(completed_keys))

if RUN_API:
    required_api_keys = {}
    if requests_df["provider"].eq("openai").any():
        required_api_keys["OPENAI_API_KEY"] = "OpenAI"
    if requests_df["provider"].eq("anthropic").any():
        required_api_keys["ANTHROPIC_API_KEY"] = "Anthropic"

    missing_keys = [key for key in required_api_keys if not read_env_value(key)[0]]
    if missing_keys:
        raise RuntimeError(
            "Missing API key(s) in .env: " + ", ".join(missing_keys) +
            ". Add them before setting RUN_API=True."
        )

    calls_done = 0
    for _, request_row in requests_df.iterrows():
        key = (
            str(request_row['item_id']),
            str(request_row['prompt_variant']),
            str(request_row['provider']),
            str(request_row['model']),
        )
        if key in completed_keys:
            continue

        base = request_row.drop(labels=['prompt']).to_dict()
        try:
            response = call_model(
                provider=request_row['provider'],
                model=request_row['model'],
                prompt=request_row['prompt'],
            )
            result = {
                **base,
                **response,
                'run_status': 'ok',
                'error': '',
                'created_at_utc': dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            }
            completed_keys.add(key)
        except Exception as exc:
            result = {
                **base,
                'category': '',
                'evidence': '',
                'raw_response': '',
                'response_id': '',
                'api_key_source': '',
                'run_status': 'error',
                'error': repr(exc),
                'created_at_utc': dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            }
            if STOP_ON_ERROR:
                results.append(result)
                save_results(results, LONG_OUTPUT_PATH)
                raise
        results.append(result)
        calls_done += 1

        if calls_done % SAVE_EVERY_N_CALLS == 0:
            save_results(results, LONG_OUTPUT_PATH)
            print(f'Saved after {calls_done} new calls -> {LONG_OUTPUT_PATH}')

        time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)

    results_df = save_results(results, LONG_OUTPUT_PATH)
    print('Final long results saved to:', LONG_OUTPUT_PATH)
else:
    print('API run disabled. Set RUN_API = True after setting OPENAI_API_KEY / ANTHROPIC_API_KEY.')
    print('Request plan is ready at:', REQUEST_PLAN_PATH)
    results_df = existing_df

if not results_df.empty:
    display(results_df.head(10))
else:
    print('No model labels yet. Run the API cell with RUN_API=True to create:', LONG_OUTPUT_PATH)

In [ ]:
# Run-level quality summary
# hit_rate = successful labels / planned labels for each model.
# completion_rate = any returned row / planned labels for each model.

summary_keys = ['prompt_variant', 'provider', 'model']
planned_summary = (
    requests_df.groupby(summary_keys, as_index=False)
    .size()
    .rename(columns={'size': 'planned_rows'})
)

if 'results_df' not in globals() or results_df.empty:
    run_summary = planned_summary.assign(
        returned_rows=0,
        ok_rows=0,
        error_rows=0,
        completion_rate=0.0,
        hit_rate=0.0,
    )
    print('No current API results yet. Summary shows planned rows only.')
else:
    returned_summary = (
        results_df.groupby(summary_keys, as_index=False)
        .agg(
            returned_rows=('item_id', 'size'),
            ok_rows=('run_status', lambda s: int(s.eq('ok').sum())),
            error_rows=('run_status', lambda s: int(s.eq('error').sum())),
        )
    )
    run_summary = planned_summary.merge(returned_summary, on=summary_keys, how='left')
    for col in ['returned_rows', 'ok_rows', 'error_rows']:
        run_summary[col] = run_summary[col].fillna(0).astype(int)
    run_summary['completion_rate'] = (run_summary['returned_rows'] / run_summary['planned_rows']).round(4)
    run_summary['hit_rate'] = (run_summary['ok_rows'] / run_summary['planned_rows']).round(4)

run_summary.to_csv(SUMMARY_OUTPUT_PATH, index=False)
print('Run summary saved to:', SUMMARY_OUTPUT_PATH)
display(run_summary)


In [ ]:
# Claude-only retry cell for rate-limit errors
# Use this after the full run if the summary shows Claude error rows.
# It preserves all ok rows, drops previous Claude non-ok rows, and retries only missing Claude classifications.

CLAUDE_RETRY_SLEEP_SECONDS = 90
CLAUDE_RETRY_MAX_ATTEMPTS = 12
CLAUDE_RETRY_WAIT_SECONDS = 180

labels_existing = pd.read_csv(LONG_OUTPUT_PATH) if LONG_OUTPUT_PATH.exists() else pd.DataFrame()
if labels_existing.empty:
    raise FileNotFoundError(f'No existing labels found at {LONG_OUTPUT_PATH}')

ok_existing = labels_existing.loc[labels_existing['run_status'].eq('ok')].copy()
existing_ok_keys = set(zip(
    ok_existing['item_id'].astype(str),
    ok_existing['prompt_variant'].astype(str),
    ok_existing['provider'].astype(str),
    ok_existing['model'].astype(str),
))

claude_requests = requests_df.loc[
    requests_df['provider'].eq('anthropic') & requests_df['model'].eq('claude-sonnet-4-5')
].copy()
claude_requests['_key'] = list(zip(
    claude_requests['item_id'].astype(str),
    claude_requests['prompt_variant'].astype(str),
    claude_requests['provider'].astype(str),
    claude_requests['model'].astype(str),
))
retry_requests = claude_requests.loc[~claude_requests['_key'].isin(existing_ok_keys)].drop(columns=['_key']).copy()

results = ok_existing.to_dict('records')
print('Existing ok rows kept:', len(ok_existing))
print('Claude retry rows:', len(retry_requests))

def save_retry_results():
    retry_df = save_results(results, LONG_OUTPUT_PATH)
    return retry_df

for _, request_row in retry_requests.iterrows():
    base = request_row.drop(labels=['prompt']).to_dict()
    last_error = None
    for attempt in range(1, CLAUDE_RETRY_MAX_ATTEMPTS + 1):
        try:
            response = call_model(
                provider=request_row['provider'],
                model=request_row['model'],
                prompt=request_row['prompt'],
            )
            result = {
                **base,
                **response,
                'run_status': 'ok',
                'error': '',
                'created_at_utc': dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            }
            results.append(result)
            save_retry_results()
            print(f"Claude ok item_id={request_row['item_id']} row_id={request_row['row_id']} ({len(results)}/400)")
            break
        except Exception as exc:
            last_error = repr(exc)
            is_rate_limit = 'HTTP 429' in last_error or 'rate_limit_error' in last_error
            if is_rate_limit and attempt < CLAUDE_RETRY_MAX_ATTEMPTS:
                wait_seconds = CLAUDE_RETRY_WAIT_SECONDS * attempt
                print(f"Claude 429 item_id={request_row['item_id']} attempt={attempt}; waiting {wait_seconds}s")
                time.sleep(wait_seconds)
                continue
            result = {
                **base,
                'category': '',
                'evidence': '',
                'raw_response': '',
                'response_id': '',
                'api_key_source': '',
                'run_status': 'error',
                'error': last_error,
                'created_at_utc': dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            }
            results.append(result)
            save_retry_results()
            print(f"Claude error item_id={request_row['item_id']} row_id={request_row['row_id']}: {last_error[:220]}")
            break
    time.sleep(CLAUDE_RETRY_SLEEP_SECONDS)

results_df = save_retry_results()
print('Claude retry finished. Current status:')
display(results_df.groupby(['provider', 'model', 'run_status']).size().reset_index(name='rows'))


In [ ]:
if not RUN_API and RESET_EXISTING_RESULTS:
    labels_long = pd.DataFrame()
    print('Dry run only: skipping wide export so old CSV rows are not treated as current results.')
elif LONG_OUTPUT_PATH.exists():
    labels_long = pd.read_csv(LONG_OUTPUT_PATH)
else:
    labels_long = pd.DataFrame()

if labels_long.empty:
    print('No current long results found yet. Run the API cell first.')
else:
    id_cols = ['item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window']

    # Wide file is only for quick side-by-side inspection.
    # The long file remains the main analysis-ready output: one article/entity/model row.
    wide_parts = run_sample_df[id_cols].copy()
    for label, group in labels_long.groupby('run_label'):
        cols = group[['item_id', 'provider', 'model', 'category', 'evidence', 'run_status', 'error']].copy()
        cols = cols.rename(columns={
            'provider': f'{label}__provider',
            'model': f'{label}__model',
            'category': f'{label}__category',
            'evidence': f'{label}__evidence',
            'run_status': f'{label}__status',
            'error': f'{label}__error',
        })
        wide_parts = wide_parts.merge(cols, on='item_id', how='left')

    wide_parts.to_csv(WIDE_OUTPUT_PATH, index=False)
    print('Wide labels saved to:', WIDE_OUTPUT_PATH)
    display(wide_parts.head(10))

In [ ]:
# Readable final wide export + retry queue
# Long CSV = audit trail, one row per article/entity/model.
# Readable wide CSV = one row per sample item, with category/evidence/status/error columns per model.

if not LONG_OUTPUT_PATH.exists():
    print('No long labels found yet; run the API cell first.')
else:
    labels_long = pd.read_csv(LONG_OUTPUT_PATH)
    status_rank = {'ok': 0, 'error': 1}
    labels_long['_status_rank'] = labels_long['run_status'].map(status_rank).fillna(9)
    labels_long = (
        labels_long.sort_values(['item_id', 'prompt_variant', 'provider', 'model', '_status_rank', 'created_at_utc'])
        .drop_duplicates(['item_id', 'prompt_variant', 'provider', 'model'], keep='first')
        .drop(columns=['_status_rank'])
    )

    readable_wide = sample_df.loc[:, [
        'item_id', 'hit_id', 'row_id', 'outlet', 'article_title', 'entity_mention', 'context_window'
    ]].rename(columns={'context_window': 'text'}).copy()

    model_order = [
        ('openai', 'gpt-5.4', 'gpt_5_4'),
        ('openai', 'gpt-5.4-mini', 'gpt_5_4_mini'),
        ('openai', 'gpt-5-mini', 'gpt_5_mini'),
        ('anthropic', 'claude-sonnet-4-5', 'claude_sonnet_4_5'),
    ]

    for provider, model, prefix in model_order:
        part = labels_long.loc[
            labels_long['provider'].eq(provider) & labels_long['model'].eq(model),
            ['item_id', 'provider', 'model', 'category', 'evidence', 'run_status', 'error']
        ].copy()
        part = part.rename(columns={
            'provider': f'{prefix}_provider',
            'model': f'{prefix}_model',
            'category': f'{prefix}_category',
            'evidence': f'{prefix}_evidence',
            'run_status': f'{prefix}_status',
            'error': f'{prefix}_error',
        })
        readable_wide = readable_wide.merge(part, on='item_id', how='left')

    retry_queue = labels_long.loc[~labels_long['run_status'].eq('ok')].copy()
    readable_wide.to_csv(READABLE_WIDE_OUTPUT_PATH, index=False)
    retry_queue.to_csv(ERRORS_TO_RETRY_PATH, index=False)

    print('Readable wide CSV saved to:', READABLE_WIDE_OUTPUT_PATH)
    print('Retry/error queue saved to:', ERRORS_TO_RETRY_PATH)
    print('Readable wide shape:', readable_wide.shape)
    print('Retry/error rows:', len(retry_queue))
    display(labels_long.groupby(['provider', 'model', 'run_status']).size().reset_index(name='rows'))
    display(readable_wide.head(10))


In [ ]:
if not RUN_API and RESET_EXISTING_RESULTS:
    print('Dry run only: skipping label analysis so old CSV rows are not treated as current results.')
elif not LONG_OUTPUT_PATH.exists():
    print('No labels to analyze yet.')
else:
    labels_long = pd.read_csv(LONG_OUTPUT_PATH)
    ok = labels_long.loc[labels_long['run_status'].eq('ok')].copy()
    print('OK labels:', len(ok), '/', len(labels_long))

    if ok.empty:
        display(labels_long['run_status'].value_counts(dropna=False))
    else:
        print('Category distribution by model:')
        display(pd.crosstab(ok['run_label'], ok['category'], margins=True))

        category_matrix = ok.pivot_table(
            index='item_id',
            columns='run_label',
            values='category',
            aggfunc='first',
        )
        display(category_matrix.head(20))

        pairwise_rows = []
        cols = list(category_matrix.columns)
        for i, left in enumerate(cols):
            for right in cols[i + 1:]:
                pair = category_matrix[[left, right]].dropna()
                agreement = float((pair[left] == pair[right]).mean()) if len(pair) else float('nan')
                pairwise_rows.append({
                    'model_a': left,
                    'model_b': right,
                    'n_overlap': len(pair),
                    'agreement': agreement,
                })
        pairwise_agreement = pd.DataFrame(pairwise_rows)
        display(pairwise_agreement.sort_values('agreement', ascending=True))

        disagreement_mask = category_matrix.nunique(axis=1, dropna=True) > 1
        disagreements = sample_df.merge(
            category_matrix.loc[disagreement_mask].reset_index(),
            on='item_id',
            how='inner',
        )
        print('Rows with model disagreement:', len(disagreements))
        display(disagreements.head(25))

In [ ]:
# Accuracy layer
# If you add a manual gold label column to the input CSV, this cell computes accuracy.
# Supported gold column names: gold_category, manual_category, true_category, label, category_gold.

gold_candidates = ['gold_category', 'manual_category', 'true_category', 'label', 'category_gold']
gold_col = next((col for col in gold_candidates if col in sample_df.columns), None)

if gold_col is None:
    print('No manual gold-label column found. Add one of these columns to the test CSV to compute accuracy:')
    print(gold_candidates)
    print('Until then, use the agreement/disagreement analysis above as the quality diagnostic.')
elif not RUN_API and RESET_EXISTING_RESULTS:
    print('Dry run only: skipping accuracy analysis so old CSV rows are not treated as current results.')
elif not LONG_OUTPUT_PATH.exists():
    print('Gold labels exist, but no model outputs found yet.')
else:
    labels_long = pd.read_csv(LONG_OUTPUT_PATH)
    ok = labels_long.loc[labels_long['run_status'].eq('ok')].copy()
    gold = sample_df[['item_id', gold_col]].rename(columns={gold_col: 'gold_category'})
    eval_df = ok.merge(gold, on='item_id', how='left')
    eval_df = eval_df.loc[eval_df['gold_category'].notna() & eval_df['gold_category'].ne('')].copy()

    if eval_df.empty:
        print('Gold column exists but contains no usable labels.')
    else:
        accuracy = (
            eval_df.assign(correct=eval_df['category'].eq(eval_df['gold_category']))
            .groupby('run_label', as_index=False)
            .agg(n=('correct', 'size'), accuracy=('correct', 'mean'))
            .sort_values('accuracy', ascending=False)
        )
        display(accuracy)

        for label, group in eval_df.groupby('run_label'):
            print('\nConfusion matrix:', label)
            display(pd.crosstab(group['gold_category'], group['category'], margins=True))